In [3]:



def balance(cost, supply, demand):
    """If total supply != total demand, add a dummy row/column with 0 cost."""
    cost = [row[:] for row in cost]
    supply = supply[:]
    demand = demand[:]
    total_s = sum(supply)
    total_d = sum(demand)

    if total_s > total_d:          # extra supply -> dummy destination
        for row in cost:
            row.append(0)
        demand.append(total_s - total_d)
        print("Note: added a dummy destination (supply > demand)")
    elif total_d > total_s:        # extra demand -> dummy source
        cost.append([0] * len(demand))
        supply.append(total_d - total_s)
        print("Note: added a dummy source (demand > supply)")
    return cost, supply, demand


def total_cost(cost, alloc):
    total = 0
    for i in range(len(cost)):
        for j in range(len(cost[0])):
            total += cost[i][j] * alloc[i][j]
    return total


# ---------- 1. North-West Corner ----------
def northwest(cost, supply, demand):
    s = supply[:]
    d = demand[:]
    rows = len(s)
    cols = len(d)
    alloc = [[0] * cols for _ in range(rows)]

    i = 0
    j = 0
    while i < rows and j < cols:
        q = min(s[i], d[j])
        alloc[i][j] = q
        s[i] -= q
        d[j] -= q
        if s[i] == 0:
            i += 1      # row finished, go down
        else:
            j += 1      # column finished, go right
    return alloc


# ---------- 2. Least Cost ----------
def least_cost(cost, supply, demand):
    s = supply[:]
    d = demand[:]
    rows = len(s)
    cols = len(d)
    alloc = [[0] * cols for _ in range(rows)]

    while sum(s) > 0 and sum(d) > 0:
        # find the cheapest cell that still has supply and demand
        best_i = -1
        best_j = -1
        best_cost = 10**9
        for i in range(rows):
            if s[i] > 0:
                for j in range(cols):
                    if d[j] > 0 and cost[i][j] < best_cost:
                        best_cost = cost[i][j]
                        best_i = i
                        best_j = j

        q = min(s[best_i], d[best_j])
        alloc[best_i][best_j] = q
        s[best_i] -= q
        d[best_j] -= q
    return alloc


# ---------- 3. Vogel's Approximation Method ----------
def penalty(values):
    """Difference between the two smallest numbers."""
    if len(values) == 1:
        return values[0]
    values = sorted(values)
    return values[1] - values[0]


def vogel(cost, supply, demand):
    s = supply[:]
    d = demand[:]
    rows = len(s)
    cols = len(d)
    alloc = [[0] * cols for _ in range(rows)]

    rows_left = list(range(rows))
    cols_left = list(range(cols))

    while len(rows_left) > 0 and len(cols_left) > 0:
        best_pen = -1
        kind = ""
        idx = -1

        # row penalties
        for i in rows_left:
            p = penalty([cost[i][j] for j in cols_left])
            if p > best_pen:
                best_pen = p
                kind = "row"
                idx = i

        # column penalties
        for j in cols_left:
            p = penalty([cost[i][j] for i in rows_left])
            if p > best_pen:
                best_pen = p
                kind = "col"
                idx = j

        # pick cheapest cell in the row/column with the biggest penalty
        if kind == "row":
            i = idx
            j = cols_left[0]
            for c in cols_left:
                if cost[i][c] < cost[i][j]:
                    j = c
        else:
            j = idx
            i = rows_left[0]
            for r in rows_left:
                if cost[r][j] < cost[i][j]:
                    i = r

        q = min(s[i], d[j])
        alloc[i][j] = q
        s[i] -= q
        d[j] -= q

        if s[i] == 0:
            rows_left.remove(i)
        else:
            cols_left.remove(j)
    return alloc


def show(name, alloc, cost):
    print("\n" + name)
    for row in alloc:
        print("  ", row)
    c = total_cost(cost, alloc)
    print("   Total cost =", c)
    return c


# ---------- Main ----------
print("=== Transportation Problem Solver ===")
n = int(input("Number of sources (rows): "))
m = int(input("Number of destinations (columns): "))

print("\nEnter cost matrix, one row per line (space separated):")
cost = []
for i in range(n):
    row = list(map(int, input("Row " + str(i + 1) + ": ").split()))
    cost.append(row)

supply = list(map(int, input("\nSupply (" + str(n) + " values): ").split()))
demand = list(map(int, input("Demand (" + str(m) + " values): ").split()))

cost, supply, demand = balance(cost, supply, demand)

results = {}
results["North-West Corner"] = show("North-West Corner", northwest(cost, supply, demand), cost)
results["Least Cost"] = show("Least Cost", least_cost(cost, supply, demand), cost)
results["Vogel (VAM)"] = show("Vogel (VAM)", vogel(cost, supply, demand), cost)

print("\n=== Comparison ===")
for name in results:
    print(name, "->", results[name])

best = min(results, key=results.get)
print("\nCheapest method:", best, "with cost", results[best])

=== Transportation Problem Solver ===
Number of sources (rows): 3
Number of destinations (columns): 3

Enter cost matrix, one row per line (space separated):
Row 1: 40 30 60
Row 2: 30 60 40
Row 3: 20 30 50

Supply (3 values): 120 180 150
Demand (3 values): 60 150 240

North-West Corner
   [60, 60, 0]
   [0, 90, 90]
   [0, 0, 150]
   Total cost = 20700

Least Cost
   [0, 120, 0]
   [0, 0, 180]
   [60, 30, 60]
   Total cost = 15900

Vogel (VAM)
   [0, 120, 0]
   [0, 0, 180]
   [60, 30, 60]
   Total cost = 15900

=== Comparison ===
North-West Corner -> 20700
Least Cost -> 15900
Vogel (VAM) -> 15900

Cheapest method: Least Cost with cost 15900


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.9 MB/s eta 0:00:00


In [ ]:
def check_jobs(jobs):
    names = []
    for j in jobs:
        if j["name"] == "":
            return "Every job needs a name."
        if j["name"] in names:
            return "Job names must be unique."
        names.append(j["name"])
        if j["proc"] <= 0:
            return "Processing time must be greater than 0."
        if j["arrival"] < 0:
            return "Arrival time cannot be negative."
        if j["priority"] < 1:
            return "Priority must be 1 or more (1 = highest)."
    return ""


# ---------- Find the order for each method ----------
def order_fcfs(jobs):
    # earliest arrival first
    return sorted(jobs, key=lambda j: (j["arrival"], j["name"]))


def order_edd(jobs):
    # earliest due date first
    return sorted(jobs, key=lambda j: (j["due"], j["arrival"], j["name"]))


def order_priority(jobs):
    # priority 1 first
    return sorted(jobs, key=lambda j: (j["priority"], j["arrival"], j["name"]))


def order_spt(jobs):
    # when the machine is free, pick the shortest job that has already arrived
    remaining = jobs[:]
    order = []
    time = 0
    while len(remaining) > 0:
        available = [j for j in remaining if j["arrival"] <= time]
        if len(available) == 0:
            # machine waits for the next job to arrive
            time = min(j["arrival"] for j in remaining)
            available = [j for j in remaining if j["arrival"] <= time]
        chosen = min(available, key=lambda j: (j["proc"], j["arrival"], j["name"]))
        order.append(chosen)
        remaining.remove(chosen)
        time = time + chosen["proc"]
    return order


# ---------- Run the jobs in the given order ----------
def run_schedule(order):
    clock = 0
    rows = []
    steps = []

    for position, j in enumerate(order, start=1):
        start = max(clock, j["arrival"])
        idle = start - clock
        completion = start + j["proc"]
        waiting = start - j["arrival"]
        turnaround = completion - j["arrival"]
        lateness = completion - j["due"]
        tardiness = max(0, lateness)

        rows.append({
            "name": j["name"], "arrival": j["arrival"], "proc": j["proc"],
            "due": j["due"], "start": start, "completion": completion,
            "waiting": waiting, "turnaround": turnaround,
            "lateness": lateness, "tardiness": tardiness,
        })

        if idle > 0:
            steps.append("Machine idle from " + str(clock) + " to " + str(start))
        steps.append(
            str(position) + ". Job " + j["name"] + ": start = max(" + str(clock) + ", "
            + str(j["arrival"]) + ") = " + str(start)
            + ", completion = " + str(start) + " + " + str(j["proc"]) + " = " + str(completion)
            + ", waiting = " + str(waiting)
            + ", lateness = " + str(lateness)
        )
        clock = completion

    n = len(rows)
    metrics = {
        "Makespan": max(r["completion"] for r in rows),
        "Avg completion": sum(r["completion"] for r in rows) / n,
        "Avg waiting": sum(r["waiting"] for r in rows) / n,
        "Avg turnaround": sum(r["turnaround"] for r in rows) / n,
        "Max lateness": max(r["lateness"] for r in rows),
        "Total tardiness": sum(r["tardiness"] for r in rows),
    }
    return rows, steps, metrics


# ---------- Print one method ----------
def show(method_name, rows, steps, metrics):
    print("\n" + "=" * 70)
    print(method_name)
    print("=" * 70)

    print("\nSteps:")
    for s in steps:
        print("  " + s)

    print("\n{:<8}{:>8}{:>8}{:>8}{:>10}{:>8}{:>10}".format(
        "Job", "Start", "Done", "Wait", "Turnarnd", "Late", "Tardy"))
    for r in rows:
        print("{:<8}{:>8g}{:>8g}{:>8g}{:>10g}{:>8g}{:>10g}".format(
            r["name"], r["start"], r["completion"], r["waiting"],
            r["turnaround"], r["lateness"], r["tardiness"]))

    print("\nMetrics:")
    for k in metrics:
        print("  " + k + " = " + str(round(metrics[k], 2)))


# ---------- Main ----------
if __name__ == "__main__":
    print("=== Single Machine Scheduling ===")
    n = int(input("Number of jobs: "))

    print("\nFor each job type: name processing_time arrival_time due_date priority")
    print("Example: J1 5 0 10 2")
    jobs = []
    for i in range(n):
        parts = input("Job " + str(i + 1) + ": ").split()
        jobs.append({
            "name": parts[0],
            "proc": float(parts[1]),
            "arrival": float(parts[2]),
            "due": float(parts[3]),
            "priority": float(parts[4]),
        })

    error = check_jobs(jobs)
    if error != "":
        print("Input error: " + error)
    else:
        all_orders = {
            "FCFS (First Come First Served)": order_fcfs(jobs),
            "SPT (Shortest Processing Time)": order_spt(jobs),
            "EDD (Earliest Due Date)": order_edd(jobs),
            "Priority Scheduling": order_priority(jobs),
        }

        all_metrics = {}
        for name in all_orders:
            rows, steps, metrics = run_schedule(all_orders[name])
            show(name, rows, steps, metrics)
            all_metrics[name] = metrics

        # ----- Comparison -----
        print("\n" + "=" * 70)
        print("COMPARISON (lower is better)")
        print("=" * 70)
        short = {"FCFS (First Come First Served)": "FCFS",
                 "SPT (Shortest Processing Time)": "SPT",
                 "EDD (Earliest Due Date)": "EDD",
                 "Priority Scheduling": "Priority"}

        print("{:<18}".format("Metric") + "".join("{:>10}".format(short[m]) for m in all_metrics))
        metric_names = list(next(iter(all_metrics.values())).keys())
        for k in metric_names:
            line = "{:<18}".format(k)
            for m in all_metrics:
                line += "{:>10.2f}".format(all_metrics[m][k])
            print(line)

        print("\nBest method for each metric:")
        for k in metric_names:
            best_value = min(all_metrics[m][k] for m in all_metrics)
            winners = [short[m] for m in all_metrics if all_metrics[m][k] == best_value]
            print("  " + k + ": " + ", ".join(winners))
